In [ ]:
import json
import os.path
import torch
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import pickle
from shapely.geometry import Point
import scienceplots
plt.style.use(['science', "no-latex"])
import random

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)


In [ ]:
kitgreen="#009682"
green="#98BF64"
kitblue="#4664AA"
grey5="#f2f2f2ff"
grey30="#b3b3b3ff"

In [ ]:
# seed = random.randint(0, 10000)
seed = 55
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Read dataset

In [ ]:
ITL3_region = gpd.read_file(r"./results/intermediate/ITL3_region.gpkg")
substations = gpd.read_file(r"./results/intermediate/substations.gpkg")

# Determine the region

In [ ]:
interested_locations = {
    # "London": {},
    "TLI3": {},
    "TLI4": {},
    "TLI5": {},
    "TLI6": {},
    "TLI7": {},
    "TLH1": {},
    "TLH2": {},
    "TLH3": {},
    "TLJ1":{},
    "TLF1":{},
    "TLF2":{},
    "TLC1":{},
    "TLC2":{},
    "TLD3":{},
    "TLD4":{},
    "TLD6":{},
    "TLG1":{},
    "TLG2":{},
    "TLE3":{},
    "TLE4":{},
}

In [ ]:
for itl2_code in interested_locations.keys():
    interested_region = ITL3_region[ITL3_region['ITL2'] == itl2_code]
    interested_substations = substations[substations['ITL2'] == itl2_code]

    interested_locations[itl2_code] = {
        "region": interested_region.reset_index(),
        "substation": interested_substations.reset_index()
    }

# Grid points

In [ ]:
from SpatialAllocation.GNN.utils.GraphBuilder import prepare_hetero_graph_from_processed, preprocess_features
from sklearn.preprocessing import StandardScaler

In [ ]:
with open(f"./results/intermediate/GridPoint_GNN/all_node_features_col.pickle", "rb") as f:
    numerical_col_names_all, categorical_col_members_all = pickle.load(f)

In [ ]:
for key in interested_locations.keys():
    graph_data_path = f"./results/intermediate/GNN/{key}_prepared_graph_data.pt"
    if os.path.exists(graph_data_path):
        print(f"Graph data for {key} already exists, skipping preparation.")
        continue
    with open(f"./results/intermediate/GridPoint/{key}_grid_points.pickle", "rb") as f:
        grid_gdf, step_size_m = pickle.load(f)
    gdf_a = grid_gdf.to_crs("EPSG:3857").copy()
    gdf_t = interested_locations[key]["substation"].to_crs("EPSG:3857").copy()
    gdf_s = interested_locations[key]["region"].to_crs("EPSG:3857").copy()


    print(f"{key} Agent contains {len(gdf_a)} points")
    print(f"{key} Target contains {len(gdf_t)} points")
    print(f"{key} Source contains {len(gdf_s)} regions")

    print("Standardizing coordinate data...")

    # 1. Extract coordinates from GeoDataFrame
    coords_a = np.array([[point.x, point.y] for point in gdf_a.centroid])
    coords_t = np.array([[point.x, point.y] for point in gdf_t.geometry])
    coords_s = np.array([[point.x, point.y] for point in gdf_s.centroid])

    # 2. Merge all coordinates for unified scaling
    #    This is a key step to ensure A and B are transformed in a unified coordinate system,
    #    preserving their relative spatial relationships
    all_coords = np.vstack([coords_a, coords_t, coords_s])

    # 3. Create and fit StandardScaler
    scaler = StandardScaler()
    scaler.fit(all_coords)

    # 4. Transform coordinates for A and B
    coords_a_scaled = scaler.transform(coords_a)
    coords_t_scaled = scaler.transform(coords_t)
    coords_s_scaled = scaler.transform(coords_s)

    # 5. Create new GeoDataFrame copies to store transformed coordinates
    #    (or update gdf_a and gdf_b directly)
    gdf_a_scaled = gdf_a.copy()
    gdf_t_scaled = gdf_t.copy()
    gdf_s_scaled = gdf_s.copy()

    gdf_a_scaled['geometry'] = [Point(x, y) for x, y in coords_a_scaled]
    gdf_t_scaled['geometry'] = [Point(x, y) for x, y in coords_t_scaled]
    gdf_s_scaled['geometry'] = [Point(x, y) for x, y in coords_s_scaled]

    print("Standardization complete.")
    # Build substation_idx index
    grid_proj = gdf_a.to_crs("EPSG:3857").copy()
    substations_proj = gdf_t.to_crs("EPSG:3857").copy()
    substations_proj = substations_proj.reset_index(drop=True)
    joined_gdf = gpd.sjoin_nearest(grid_proj, substations_proj[['geometry']], how="left", distance_col=None)
    joined_gdf.rename(columns={'index_right': 'substation_idx'}, inplace=True)
    gdf_a_scaled['substation_idx'] = joined_gdf['substation_idx']
    print("Completed building substation_idx index.")

    # final_features_a = preprocess_features(gdf_a[["landuse"]])
    final_features_a = preprocess_features(gdf_a, numerical_col_names_all, categorical_col_members_all)
    final_features_t = preprocess_features(gdf_t[[]])
    final_features_s = preprocess_features(gdf_s[["residential_percent", "commercial_percent", "industrial_percent", "agricultural_percent", "others_percent"]])
    # final_features_s = preprocess_features(gdf_s[[]])
    graph_data = prepare_hetero_graph_from_processed(gdf_s_scaled, gdf_a_scaled, gdf_t_scaled, final_features_s, final_features_a, relation_column='ITL3')
    torch.save(graph_data, graph_data_path)

    print(f"Graph data for {key} saved to {graph_data_path}.")